# **Mount Drive**

# **Download Data & Extract & Save as CSV**

In [ ]:
import requests
import numpy as np
years = np.arange(2008,2024)

for year in years:
  year = str(year)
  url = f'https://www.rma.usda.gov/-/media/RMA/Cause-Of-Loss/Summary-of-Business-with-Month-of-Loss/colsom_{year}.ashx?la=en'
  response = requests.get(url)
  filename = f'<DATA_ROOT>/InsuranceData/zipFiles/colsom_{str(year)}.zip'

  if response.status_code == 200:
      with open(filename, 'wb') as file:
          file.write(response.content)
      print(f"File downloaded successfully as '{filename}'")
  else:
      print(f"Failed to download the file: {response.status_code}")


In [ ]:
import zipfile
from glob import glob
zip_dir = '<DATA_ROOT>/InsuranceData/zipFiles/*.zip'

zip_files = sorted(glob(zip_dir))
for f in zip_files:
  print(f)
  extract_dir = '<DATA_ROOT>/InsuranceData/txtFiles/'
  with zipfile.ZipFile(f, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)


In [ ]:
import pandas as pd
from glob import glob
crops = ['HAY', 'BARLEY', 'OATS', 'CORN', 'COTTON', 'SORGHUM', 'SOYBEANS','WHEAT']

cols = [
    'Year_Identifier','State_Code','State_Abb','County_Code','County_Name','Commodity_Code','Commodity_Name',
    'Insurance_Plan_Code','Insurance_Plan_Name_Abb','Coverage_Category','Stage_Code','Cause_of_Loss_Code',
    'Cause_of_Loss_Desc','Month_of_Loss','Month_of_Loss_Name','Year_of_Loss','Policies_Earning_Premium',
    'Policies_Indemnified','Net_Planted_Quantity','Net_Endorsed_Acres','Liability','Total_Premium',
    'Producer_Paid_Premium','Subsidy','State_Private_Subsidy','Additional_Subsidy','EFA_Premium_Discount',
    'Net_Determined_Quantity','Indemnity_Amount','Loss_Ratio'
]
in_path = '<DATA_ROOT>/InsuranceData/txtFiles/*.txt'
out_path = '<DATA_ROOT>/InsuranceData/csvFiles/'
files = glob(in_path)
for f in files:
  fname = f.split('/')[-1].split('.')[0]
  fpath = out_path + fname + '.csv'
  df = pd.read_csv(f, delimiter='|',header=None)

  df.columns = cols
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

  cond1 = df['Commodity_Name'].str.contains('Sorghum', case=False, na=False)
  df.loc[cond1,'Commodity_Name'] = 'Sorghum'
  cond1 = df['Commodity_Name'].str.contains('Alfalfa', case=False, na=False)
  df.loc[cond1,'Commodity_Name'] = 'Hay'

  df.Commodity_Name = df.Commodity_Name.str.upper()
  crops = ['HAY', 'BARLEY', 'OATS', 'CORN', 'COTTON', 'SORGHUM', 'SOYBEANS','WHEAT']
  df = df[df['Commodity_Name'].isin(crops)]

  df.to_csv(fpath)


In [ ]:

in_path = '<DATA_ROOT>/InsuranceData/csvFiles/*.csv'
files = glob(in_path)
list_df = []
for f in files:
  df = pd.read_csv(f)
  list_df.append(df)

df_merge = pd.concat(list_df)
df_merge = df_merge.drop(columns='Unnamed: 0')
df_merge.to_csv('<DATA_ROOT>/InsuranceData/InsuranceData.csv',index=False)

In [ ]:
cols = ['State_Code' , 'County_Code',  'Commodity_Name' , 'Insurance_Plan_Name_Abb' , ]
df_merge.columns

In [ ]:
df_merge.columns

In [ ]:
cols = [
    'State_Code', 'County_Code','Commodity_Name',
    'Year_of_Loss','Month_of_Loss',
    'Cause_of_Loss_Code', 'Cause_of_Loss_Desc',
    'Net_Planted_Quantity', 'Net_Endorsed_Acres',
    'Liability' , 'Subsidy',
    'Net_Determined_Quantity','Indemnity_Amount']
df_merge[cols]